# Orbit Wars PPO theta.4 PFSP Training

T4 GPU, 50k step, warm-start from ppo_v3_theta3.zip

In [ ]:
import os
import shutil
import subprocess

# Approach: GitHub clone for source + Kaggle dataset for weights
# (= repo is public, weights are .gitignore-excluded)

print("=== /kaggle/input/ contents (= attached datasets) ===")
subprocess.run(["ls", "-la", "/kaggle/input/"], check=False)

REPO_DIR = "/kaggle/working/orbit-wars"
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ykaya-jp/orbit-wars.git", REPO_DIR],
        check=True,
    )
    print("cloned repo to", REPO_DIR)

# Locate dataset for ppo_v3_theta3 weights (= auto-extracted from .zip → directory)
# Try common dataset paths
candidates = [
    "/kaggle/input/orbit-wars-source",
    "/kaggle/input/orbit-wars-source-1",
]
DATASET_DIR = None
for c in candidates:
    if os.path.isdir(c):
        DATASET_DIR = c
        break
if DATASET_DIR is None:
    # Search any /kaggle/input subdirs
    subdirs = [
        d for d in os.listdir("/kaggle/input") if os.path.isdir(os.path.join("/kaggle/input", d))
    ]
    if subdirs:
        DATASET_DIR = os.path.join("/kaggle/input", subdirs[0])
        print(f"fallback: using {DATASET_DIR}")
    else:
        raise RuntimeError("Dataset not attached! Add it manually via UI.")

print(f"\nDATASET_DIR: {DATASET_DIR}")
subprocess.run(["ls", "-la", DATASET_DIR], check=False)

# Re-zip ppo_v3_theta3/ directory back into .zip (= Kaggle auto-extract reverse)
PPO_DIR = os.path.join(DATASET_DIR, "agents/proxy/ppo_v3_theta3")
PPO_ZIP = "/kaggle/working/orbit-wars/agents/proxy/ppo_v3_theta3.zip"
os.makedirs(os.path.dirname(PPO_ZIP), exist_ok=True)
if os.path.isdir(PPO_DIR):
    shutil.make_archive(PPO_ZIP.replace(".zip", ""), "zip", PPO_DIR)
    print(f"\nre-zipped {PPO_DIR} → {PPO_ZIP} ({os.path.getsize(PPO_ZIP) / 1024**2:.1f} MB)")
elif os.path.exists(os.path.join(DATASET_DIR, "agents/proxy/ppo_v3_theta3.zip")):
    shutil.copy(os.path.join(DATASET_DIR, "agents/proxy/ppo_v3_theta3.zip"), PPO_ZIP)
    print(f"copied zip from dataset → {PPO_ZIP}")
else:
    raise RuntimeError(f"ppo_v3_theta3 weights not found in {DATASET_DIR}")

# Verify critical files
print("\n=== Critical files check ===")
for rel in [
    "agents/proxy/ppo_v3_theta3.zip",
    "tools/train_ppo_pfsp.py",
    "src/orbit_wars/grid_agent.py",
    "agents/proxy/grid_il_lakhindar.py",
    "submissions/build_konbu_topk1/main.py",
    "submissions/build_rudra_topk1_proper/main.py",
]:
    full = os.path.join(REPO_DIR, rel)
    print(f'  {rel}: {"OK" if os.path.exists(full) else "MISSING"}')

In [ ]:
!pip install -q sb3-contrib==2.5.0 stable-baselines3==2.5.0 kaggle_environments gymnasium==0.29.1
!python -c 'import torch; print("cuda:", torch.cuda.is_available(), "device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")'

In [ ]:
import os
import subprocess

REPO_DIR = "/kaggle/working/orbit-wars"
os.chdir(REPO_DIR)
POOL_DIR = "/kaggle/working/ppo_pfsp_pool"
os.makedirs(POOL_DIR, exist_ok=True)
OUTPUT = "/kaggle/working/ppo_v4_theta4_kaggle.zip"

# Lakhindar weight: copy from dataset if grid_il_lakhindar.pt missing in repo
lakhindar_pt = os.path.join(REPO_DIR, "agents/proxy/grid_il_lakhindar.pt")
if not os.path.exists(lakhindar_pt):
    # search dataset
    for d in os.listdir("/kaggle/input"):
        src = os.path.join("/kaggle/input", d, "agents/proxy/grid_il_lakhindar.pt")
        if os.path.exists(src):
            import shutil

            shutil.copy(src, lakhindar_pt)
            print(f"copied lakhindar weight from {src}")
            break

cmd = [
    "python",
    "-m",
    "tools.train_ppo_pfsp",
    "--total-timesteps",
    "50000",
    "--n-envs",
    "4",
    "--n-steps",
    "512",
    "--batch-size",
    "64",
    "--device",
    "cuda",
    "--warm-start",
    "agents/proxy/ppo_v3_theta3.zip",
    "--external-opponents",
    "agents/proxy/grid_il_lakhindar.py",
    "submissions/build_konbu_topk1/main.py",
    "submissions/build_rudra_topk1_proper/main.py",
    "--pool-max",
    "4",
    "--save-interval",
    "5000",
    "--pool-dir",
    POOL_DIR,
    "--self-play-prob",
    "0.6",
    "--external-prob",
    "0.3",
    "--reward-shaping",
    "--output",
    OUTPUT,
]
print("cmd:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import glob
import os

OUTPUT = "/kaggle/working/ppo_v4_theta4_kaggle.zip"
POOL_DIR = "/kaggle/working/ppo_pfsp_pool"
print("=== Final output ===")
if os.path.exists(OUTPUT):
    print(f"  {OUTPUT}: {os.path.getsize(OUTPUT) / 1024**2:.1f} MB")
else:
    print(f"  MISSING: {OUTPUT}")
print("=== Pool checkpoints ===")
for p in sorted(glob.glob(f"{POOL_DIR}/*.zip")):
    print(f"  {os.path.basename(p)}: {os.path.getsize(p) / 1024**2:.1f} MB")